In [1]:
import data_loader, audio_preprocessing ,utils_audio_PD_project
import os
import pandas as pd
import numpy as np

c:\Users\Marcos\anaconda3\lib\site-packages\paramiko\transport.py:219: CryptographyDeprecationWarning: Blowfish has been deprecated
  "class": algorithms.Blowfish,


In [10]:
import os
import pandas as pd
pd.set_option('max_colwidth', 400)

# Define the mapping for labels based on folder names
mapping_dict = {
    'HC': 0,
    'PD': 1
}

def load_audio_data(root_directory, metadata_df, start_with=None, exact_name=None, 
                    patient_type_mapping=mapping_dict, audio_extensions=['.wav']):
    """
    Loads audio file paths and labels for patients, mapping to their IDs from metadata.

    Args:
    - root_directory (str): Directory containing 'HC' and 'PD' folders.
    - metadata_df (pd.DataFrame): DataFrame with 'name', 'surname', and 'ID' columns.
    - start_with (str, optional): Filter files starting with this string.
    - exact_name (str, optional): Filter files with this exact name.
    - patient_type_mapping (dict): Maps folder names to labels.
    - audio_extensions (list): Audio file extensions to consider.

    Returns:
    - pd.DataFrame: Contains 'Patient', 'Label', 'File_Path', 'ID'.
    """
    
    data = []
    
    for patient_type in patient_type_mapping:
        category_dir = os.path.join(root_directory, patient_type)
        if not os.path.exists(category_dir):
            continue
        
        for patient_folder in os.listdir(category_dir):
            patient_dir = os.path.join(category_dir, patient_folder)
            if not os.path.isdir(patient_dir):
                continue
            
            # Split folder into name and surname
            parts = patient_folder.strip().split()
            if len(parts) < 2:
                print(f"Skipping invalid folder: {patient_folder}")
                continue
            name, surname = ' '.join(parts[:-1]).strip(), parts[-1].strip()
            
            # Find matching patient in metadata
            mask = (metadata_df['name'].str.strip().str.lower() == name.lower()) & \
            (metadata_df['surname'].str.strip().str.lower() == surname.lower())
            
            matches = metadata_df[mask]
            
            if matches.empty:
                print(f"No metadata match for {patient_folder}. Skipping.")
                continue
            patient_id = str(matches.iloc[0]['ID'])
            label = patient_type_mapping[patient_type]
            
            # Process each audio file
            for file in os.listdir(patient_dir):
                if not any(file.lower().endswith(ext) for ext in audio_extensions):
                    continue
                if start_with and not file.lower().startswith(start_with.lower()):
                    continue
                if exact_name and file != exact_name:
                    continue
                
                file_path = os.path.join(patient_dir, file)
                data.append({
                    'Patient': patient_folder,
                    'Label': label,
                    'File_Path': file_path,
                    'ID': patient_id
                })
    
    df = pd.DataFrame(data)
    if df.empty:
        print("No files found.")
    else:
        print(f"Loaded {len(df)} files. Label distribution:\n{df['Label'].value_counts()}")
    return df

In [3]:
# Define the root directory containing the patient types
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV'

# Define the patient categories (HC, NFC, AC, PD)
patient_types = ['HC','PD']


# Define the patient type mapping
patient_type_mapping = {
    'HC': 0,
    'PD': 1
}

# Define audio extensions
audio_extensions = ['.wav']

In [4]:
metadata_df = pd.read_csv(r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\metadata_IVP_df.csv', sep=';')
metadata_df.head(10)

,name,surname,Label type,Num,ID,Label,sex,age,from,time1,CPS1,time2,CPS2,time3,CPS3,Unnamed: 15
0,Brigida,C,HC,1,IPV_HC_1,0,F,69,Bari,"57,12","9,07","49,99","10,36","47,11","5,96",NaN
1,Michele,G,HC,2,IPV_HC_2,0,M,68,Bari,"59,55","8,7","55,08","9,33","43,47","6,46",NaN
2,Angela,C,HC,3,IPV_HC_3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN
3,Leonarda,F,HC,4,IPV_HC_4,0,F,60,Bari,"59,49","7,88","54,97","8,53","43,96","6,39",NaN
4,Antonietta,P,HC,5,IPV_HC_5,0,F,61,Bari,"66,92","7,74","53,89","9,61","43,77","6,42",NaN
5,Angela,G,HC,6,IPV_HC_6,0,F,63,Bari,"70,3","6,69","71,26","6,6","44,8","6,27",NaN
6,Vito,A,HC,7,IPV_HC_7,0,M,68,Bari,"58,57","8,74","51,26","9,85","42,02","6,69",NaN
7,Luigi,P,HC,8,IPV_HC_8,0,M,76,Bari,"56,09","9,24","53,15","9,75","49,15","5,72",NaN
8,Antonio,C,HC,9,IPV_HC_9,0,M,77,Bari,"67,97","7,59","59,75","8,27","67,33","4,17",NaN
9,Teresa,M,HC,10,IPV_HC_10,0,F,63,Bari,"75,88","6,68","62,73","7,94","54,65","5,03",NaN


In [6]:
df = load_audio_data(root_directory, metadata_df, start_with=None)
df.head()

Loaded 725 files. Label distribution:
Label
1    437
0    288
Name: count, dtype: int64


,Patient,Label,File_Path,ID
0,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\B1ACNAGRER49F210320170916.wav,IPV_HC_3
1,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\B2ACNAGRER49F210320170919.wav,IPV_HC_3
2,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\D1ACNAGRER49F210320171134.wav,IPV_HC_3
3,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\D2ACNAGRER49F210320171135.wav,IPV_HC_3
4,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\FB1ACNAGRER49F210320170926.wav,IPV_HC_3


In [7]:
df['ID']=df['ID'].astype(str)
metadata_df['ID']=metadata_df['ID'].astype(str)

In [8]:
df_merge = pd.merge(df, metadata_df, left_on='ID', right_on='ID', how='left')
df_merge.head()

,Patient,Label_x,File_Path,ID,name,surname,Label type,Num,Label_y,sex,age,from,time1,CPS1,time2,CPS2,time3,CPS3,Unnamed: 15
0,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\B1ACNAGRER49F210320170916.wav,IPV_HC_3,Angela,C,HC,3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN
1,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\B2ACNAGRER49F210320170919.wav,IPV_HC_3,Angela,C,HC,3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN
2,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\D1ACNAGRER49F210320171134.wav,IPV_HC_3,Angela,C,HC,3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN
3,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\D2ACNAGRER49F210320171135.wav,IPV_HC_3,Angela,C,HC,3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN
4,ANGELA C,0,C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV\HC\ANGELA C\FB1ACNAGRER49F210320170926.wav,IPV_HC_3,Angela,C,HC,3,0,F,68,Bari,"55,97","9,25","53,01","9,77","51,98","5,41",NaN


# Saving pre-processed audio data into ".npy" files

## All audio

In [11]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV'
output_folder = r'D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_all_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)


start_with_names = ['B', 'FB', 'PR', 'V', 'D']

# Iterate through 'start_with_names' values
for start_with in start_with_names:
    print(f"Processing audio files starting with: {start_with}")

    # Load audio data
    df = load_audio_data(root_directory, metadata_df=metadata_df, start_with=start_with, exact_name=None)

    # Preprocess and split audio
    audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
        df, start_time = 0, chunk_duration = 5, max_duration = None, target_sr = 16000, 
        remove_silence = True, top_db=25, silence_duration= 1, file_path_column='File_Path', 
        ID_column= 'ID', overlap= 1, min_chunk_length=0.8,
    )


    # Save arrays in the subfolder (using the modified file name)
    np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{start_with}_IPV.npy"), audio_segments)
    np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{start_with}_IPV.npy"), labels_np)
    np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{start_with}_IPV.npy"), patient_ids)


    print(f"Arrays saved successfully in {output_folder}!")

print("All audio files processed and saved.")

Processing audio files starting with: B
Loaded 88 files. Label distribution:
Label
1    52
0    36
Name: count, dtype: int64
Total chunks generated: 1333
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_all_audio!
Processing audio files starting with: FB
Loaded 44 files. Label distribution:
Label
1    26
0    18
Name: count, dtype: int64
Total chunks generated: 278
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_all_audio!
Processing audio files starting with: PR
Loaded 46 files. Label distribution:
Label
1    28
0    18
Name: count, dtype: int64
Total chunks generated: 554
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_all_audio!
Processing audio files starting with: V
Loaded 455 files. Label distribution:
Label
1    275
0    180
Name: count, dtype: int64
Total chunks generated: 918
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_all_audio!
Processing audio f

## Mean audio

Audio files will be as long as the mean of the audio

In [12]:
# Define root directory and output folder
root_directory = r'C:\Users\Marcos\Documents\IDIVAL\Proyectos\Audio_Parkinson\Italian_PV'
output_folder = r'D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio'

# Create the output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

# Define the exercise sets and their corresponding split parameters
exercise_sets = [
    # (description, exact_names_values, start_time, chunk_duration, max_duration)
    ("1 audio of 5 seconds", ['V', 'D'], 1, 5, 6),
    ("6 audios of 5 seconds", ['B', 'FB', 'PR'], 0, 5, 35),
]

for desc, start_with_values, start_time, chunk_duration, max_duration in exercise_sets:
    print(f"\nProcessing set: {desc}")
    for start_with in start_with_values:
        print(f"  Processing audio files: {start_with}")
        # Load audio data
        df = load_audio_data(root_directory, metadata_df=metadata_df, start_with=start_with, exact_name=None)

        audio_segments, labels_np, patient_ids = audio_preprocessing.execute_preprocess_and_split(
            df,
            start_time=start_time,
            chunk_duration=chunk_duration,
            max_duration=max_duration,
            target_sr=16000,
            remove_silence=True,
            top_db=25,
            silence_duration=1,
            file_path_column='File_Path',
            ID_column= 'ID',
            overlap=1,
            min_chunk_length=0.8,
        )

        np.save(os.path.join(output_folder, f"audio_segments_5s_with_1s_overlap_{start_with}_IPV.npy"), audio_segments)
        np.save(os.path.join(output_folder, f"labels_5s_with_1s_overlap_{start_with}_IPV.npy"), labels_np)
        np.save(os.path.join(output_folder, f"patient_ids_5s_with_1s_overlap_{start_with}_IPV.npy"), patient_ids)
        print(f"Arrays saved successfully in {output_folder}!")
        
print("All audio files processed and saved.")


Processing set: 1 audio of 5 seconds
  Processing audio files: V
Loaded 455 files. Label distribution:
Label
1    275
0    180
Name: count, dtype: int64
Total chunks generated: 433
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio!
  Processing audio files: D
Loaded 92 files. Label distribution:
Label
1    56
0    36
Name: count, dtype: int64
Total chunks generated: 54
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio!

Processing set: 6 audios of 5 seconds
  Processing audio files: B
Loaded 88 files. Label distribution:
Label
1    52
0    36
Name: count, dtype: int64
Total chunks generated: 704
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio!
  Processing audio files: FB
Loaded 44 files. Label distribution:
Label
1    26
0    18
Name: count, dtype: int64
Total chunks generated: 263
Arrays saved successfully in D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_

# Checking labels and ids

In [18]:
# Define directories for both datasets
dir_1 = r'D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio'

exercise_names = ['B_IPV', 'D_IPV', 'FB_IPV', 'PR_IPV', 'V_IPV']
# Load both datasets
all_patient_ids_IPV, all_labels_IPV, all_audio_segments_IPV, _ = utils_audio_PD_project.load_processed_audio_dataset(dir_1, 0, GRL=False, exercise_names = exercise_names) 
# all_patient_ids, all_labels, all_audio_segments, _ = utils_audio_PD_project.load_processed_audio_dataset(dir_1, 0, GRL=False, exercise_names = ["uy"])


# Check final dataset sizes
print(f"Total audio segments: {len(all_audio_segments_IPV)}")
print(f"Total labels: {len(all_labels_IPV)}")
print(f"Total patient IDs: {len(all_patient_ids_IPV)}")

# Validate label distribution
print(f"Final label distribution: {np.bincount(all_labels_IPV)}")

Loading dataset from: D:\processed_audio\HC_vs_PD_5s_with_1s_overlap_IPV_mean_audio
Selected exercises: ['B_IPV', 'D_IPV', 'FB_IPV', 'PR_IPV', 'V_IPV']
Sorted Patient ID files: ['patient_ids_5s_with_1s_overlap_B_IPV.npy', 'patient_ids_5s_with_1s_overlap_D_IPV.npy', 'patient_ids_5s_with_1s_overlap_FB_IPV.npy', 'patient_ids_5s_with_1s_overlap_PR_IPV.npy', 'patient_ids_5s_with_1s_overlap_V_IPV.npy']
Sorted Label files: ['labels_5s_with_1s_overlap_B_IPV.npy', 'labels_5s_with_1s_overlap_D_IPV.npy', 'labels_5s_with_1s_overlap_FB_IPV.npy', 'labels_5s_with_1s_overlap_PR_IPV.npy', 'labels_5s_with_1s_overlap_V_IPV.npy']
Sorted Audio files: ['audio_segments_5s_with_1s_overlap_B_IPV.npy', 'audio_segments_5s_with_1s_overlap_D_IPV.npy', 'audio_segments_5s_with_1s_overlap_FB_IPV.npy', 'audio_segments_5s_with_1s_overlap_PR_IPV.npy', 'audio_segments_5s_with_1s_overlap_V_IPV.npy']
Total audio segments: 1822
Total labels: 1822
Total patient IDs: 1822
Final label distribution: [ 736 1086]


In [19]:
all_patient_ids_IPV

array(['IPV_HC_3', 'IPV_HC_3', 'IPV_HC_3', ..., 'IPV_PD_10', 'IPV_PD_10',
       'IPV_PD_10'], dtype='<U9')

In [20]:
all_patient_ids_IPV

array(['IPV_HC_3', 'IPV_HC_3', 'IPV_HC_3', ..., 'IPV_PD_10', 'IPV_PD_10',
       'IPV_PD_10'], dtype='<U9')